# Train the Bragi tag heads (WP-1.6, ADR-0011)

Reproducible notebook for `services/workers/embed/training/train_heads.py`: it trains one small
MLP per category (genre, mood, instrument) on **CLAP song embeddings** and writes the
`<category>.npz` files `analysis.tag` loads from `BRAGI_TAG_HEADS` (04 §6.2).

**Not run for WP-1.6.** No GPU, no CLAP weights and no labelled dataset are available in the
implementation environment, so every number below is illustrative and no gate is decided here.
See `README.md` for the dataset (`MTG-Jamendo`, CC — re-check the per-subset licence) and the
exact commands.

In [ ]:
from pathlib import Path

from bragi_analysis.embedding.clap import CLAPEmbedder
from bragi_analysis.tagging import taxonomy

TAXONOMY = Path("packages/schemas/analysis/taxonomy.json")
DATASET = Path("/data/jamendo.jsonl")
OUT = Path("/models/bragi-heads")
vocabulary = taxonomy.load(TAXONOMY)
print("labels per category:", {c: len(vocabulary.entries(c)) for c in taxonomy.CATEGORIES})
embedder = CLAPEmbedder.load()  # refuses an unverified/missing checkpoint (MODELS.md)
print("embedder:", embedder.model, embedder.dim)

In [ ]:
# The training itself lives in the script so the run is one command and one diffable file.
import shutil
import subprocess

uv = shutil.which("uv")
if uv is None:
    raise RuntimeError("uv is not on PATH")
subprocess.run(  # noqa: S603
    [
        uv,
        "run",
        "--directory",
        "services/workers",
        "--extra",
        "clap",
        "python",
        "services/workers/embed/training/train_heads.py",
        "--dataset",
        str(DATASET),
        "--taxonomy",
        str(TAXONOMY),
        "--out",
        str(OUT),
        "--epochs",
        "40",
        "--hidden",
        "256",
        "--seed",
        "0",
    ],
    check=True,
)

In [ ]:
# Verify the heads load the way the worker loads them (bragi_analysis.tagging.zeroshot).
import numpy as np

from bragi_analysis.tagging.zeroshot import classify, load_heads

heads = load_heads(OUT)
print("trained categories:", sorted(heads))
probe = np.zeros(embedder.dim)
probe[0] = 1.0
for category, head in heads.items():
    scores = classify(head, probe)
    top = int(np.argmax(scores))
    print(category, head.labels[top], round(float(scores[top]), 4))

## The gate this notebook feeds

`analysis.tag` with `BRAGI_TAG_HEADS` set replaces the zero-shot fallback for genre/mood/
instrument. On the (owner-supplied) eval set, score the **top-3 genre agreement ≥ 0.70** gate
(04 §8). Until the eval set exists, the gate is `PENDING` (Q32) and nothing here is a product
claim. The deterministic-fallback tests in `analysis/tests/test_tagging.py` prove the machinery.